# 04 — V2 Training (Nutrient-aware encoder)

v1 MVP에서 한 가지만 바꿈: encoder가 영양소 (7d) 를 ingredient 임베딩에 additive하게 인젝션.
decoder g vector 입력과 L_health는 v1 MVP랑 동일 → "encoder 인젝션 효과"만 isolation.

T4 기준 약 2시간. 4개 g 조건 한 번에 평가됨.

**저장**: `{OUTPUT_DIR}/best_v2.pt`, `{OUTPUT_DIR}/test_predictions_v2_{{auto,1_0,0_1,1_1}}.json`

## 환경 설정

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch_geometric pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR = f'{PROJECT_ROOT}/code'
DATA_RAW = f'{PROJECT_ROOT}/data_raw'
DATA_DIR = f'{PROJECT_ROOT}/data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs'

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)

print(f'Working dir: {os.getcwd()}')
!ls

In [ ]:
import torch, torch_geometric
print(f'PyTorch: {torch.__version__}')
print(f'PyG    : {torch_geometric.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu"})')

In [ ]:
# Load NUM_TOTAL / NUM_ING from data_meta.json (created by 01_setup_data.ipynb)
import json
with open(f'{DATA_DIR}/data_meta.json') as f:
    META = json.load(f)
NUM_TOTAL = META['num_total_nodes']
NUM_ING   = META['num_ingredients']
print(f'NUM_TOTAL={NUM_TOTAL}  NUM_ING={NUM_ING}')

## 학습 + 평가

In [ ]:
# V2 학습 + 4개 g 조건 평가 (한 번에)
# 7개 nutrient key (calorie_kcal, fat_g, saturated_fat_g, carbohydrate_g,
# sugar_g, protein_g, sodium_mg) 중 usda_mapping.json에 없는 건 zero column으로 처리됨
# (학습 시작 시 WARNING 메시지 찍힘).
!python train_v2.py \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --num_total_nodes {NUM_TOTAL} --num_ingredients {NUM_ING} \
    --batch_size 64 --max_epochs 200 --patience 10 \
    --lambda_h 1.0 --margin 0.5 --tau_percentile 50 \
    --test_g_overrides auto 1_0 0_1 1_1